# J4 Hybrid Pipeline — PCA + Autoencoder + Logistic Regression (Colab GPU)

**Pipeline**: Raw data → J3 Feature Engineering (~105 features) → PCA + Autoencoder → Regularized LogReg

⚡ **Runtime → Change runtime type → GPU** pour accélérer l'autoencoder

In [ ]:
# 0. Setup & Upload Data
import os

# Mount Google Drive (alternative: upload files directly)
from google.colab import drive
drive.mount('/content/drive')

# Set data path — adjust to your Drive path
DATA_DIR = '/content/drive/MyDrive/QRT-ChallengeENS/Data'
# Or upload manually:
# from google.colab import files
# uploaded = files.upload()  # Upload X_train.csv, y_train.csv, X_test.csv
# DATA_DIR = '/content'

print('Files in DATA_DIR:', os.listdir(DATA_DIR))

In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
import warnings
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. J3 Feature Engineering (inlined)
# ══════════════════════════════════════════════════════════════

class FeatureGenerator:
    def fit(self, X, y=None): return self
    def transform(self, X): return X

class RawFeatureGenerator(FeatureGenerator):
    def __init__(self, cols): self.cols = cols
    def transform(self, X): return X[self.cols].copy()

class RollingStatFeatureGenerator(FeatureGenerator):
    def __init__(self, cols, windows, operations=['mean', 'std']):
        self.cols, self.windows, self.operations = cols, windows, operations
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            avail = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(avail) < w: continue
            if 'mean' in self.operations: X_new[f'RET_MEAN_{w}'] = X[avail].mean(axis=1)
            if 'std' in self.operations:  X_new[f'RET_STD_{w}'] = X[avail].std(axis=1)
            if 'min' in self.operations:  X_new[f'RET_MIN_{w}'] = X[avail].min(axis=1)
            if 'max' in self.operations:  X_new[f'RET_MAX_{w}'] = X[avail].max(axis=1)
            if 'skew' in self.operations: X_new[f'RET_SKEW_{w}'] = X[avail].skew(axis=1)
            if 'kurt' in self.operations: X_new[f'RET_KURT_{w}'] = X[avail].kurt(axis=1)
        return X_new

class MomentumGenerator(FeatureGenerator):
    def __init__(self, windows=[(1,5),(1,20),(5,20)]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for short, long in self.windows:
            cs, cl = f'RET_{short}', f'RET_{long}'
            if cs in X.columns and cl in X.columns:
                X_new[f'MOM_POINT_{short}_{long}'] = X[cs] - X[cl]
            vs = [f'RET_{i}' for i in range(1, short+1) if f'RET_{i}' in X.columns]
            vl = [f'RET_{i}' for i in range(1, long+1) if f'RET_{i}' in X.columns]
            if len(vs)==short and len(vl)==long:
                X_new[f'MOM_MA_{short}_{long}'] = X[vs].mean(axis=1) - X[vl].mean(axis=1)
        return X_new

class VolatilityRatioGenerator(FeatureGenerator):
    def __init__(self, windows=[(5,20)]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for short, long in self.windows:
            vs = [f'RET_{i}' for i in range(1, short+1) if f'RET_{i}' in X.columns]
            vl = [f'RET_{i}' for i in range(1, long+1) if f'RET_{i}' in X.columns]
            if len(vs)==short and len(vl)==long:
                ss, sl = X[vs].std(axis=1), X[vl].std(axis=1)
                X_new[f'VOL_RATIO_{short}_{long}'] = ss / (sl + 1e-9)
                if 'RET_1' in X.columns:
                    X_new[f'SHARPE_PROXY_{long}'] = X['RET_1'] / (sl + 1e-9)
        return X_new

class ShortTermInteractionGenerator(FeatureGenerator):
    def __init__(self, max_lag=10): self.max_lag = max_lag
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        cumul = 0
        for i in range(1, self.max_lag+1):
            rc, vc = f'RET_{i}', f'SIGNED_VOLUME_{i}'
            if rc in X.columns and vc in X.columns:
                term = X[rc] * X[vc]
                X_new[f'RET_x_VOL_{i}'] = term
                if i <= 5: cumul += term
        X_new['CUMUL_FLOW_5'] = cumul
        if 'MEDIAN_DAILY_TURNOVER' in X.columns:
            for i in range(1, 6):
                rc, vc = f'RET_{i}', f'SIGNED_VOLUME_{i}'
                if rc in X.columns and vc in X.columns:
                    X_new[f'RET_VOL_NORM_{i}'] = (X[rc]*X[vc]) / (X['MEDIAN_DAILY_TURNOVER']+1e-9)
        return X_new

class HigherOrderStatsGenerator(FeatureGenerator):
    def __init__(self, windows=[5,10,20]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            avail = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(avail) < w: continue
            X_new[f'RET_SKEW_{w}'] = X[avail].skew(axis=1)
            X_new[f'RET_KURT_{w}'] = X[avail].kurt(axis=1)
        return X_new

class SignFlipGenerator(FeatureGenerator):
    def __init__(self, windows=[5,10,20]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            avail = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(avail) < w: continue
            signs = np.sign(X[avail].values)
            X_new[f'SIGN_FLIPS_{w}'] = np.sum(np.diff(signs, axis=1) != 0, axis=1)
            X_new[f'POS_RATIO_{w}'] = np.mean(signs > 0, axis=1)
            streak = np.ones(len(X))
            for i in range(1, signs.shape[1]):
                streak += (signs[:,i] == signs[:,0]).astype(float) * (streak == i).astype(float)
            X_new[f'STREAK_{w}'] = streak
        return X_new

class VolumeFlowGenerator(FeatureGenerator):
    def __init__(self, windows=[5,10]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            av = [f'SIGNED_VOLUME_{i}' for i in range(1, w+1) if f'SIGNED_VOLUME_{i}' in X.columns]
            ar = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            if len(av) >= 2:
                vols = X[av]
                X_new[f'VOL_IMBALANCE_{w}'] = vols.sum(axis=1)
                X_new[f'VOL_STD_{w}'] = vols.std(axis=1)
                half = len(av) // 2
                X_new[f'VOL_ACCEL_{w}'] = vols.iloc[:,:half].sum(axis=1) - vols.iloc[:,half:].sum(axis=1)
            if len(ar)==w and len(av)==w:
                rv, vv = X[ar].values, np.abs(X[av].values) + 1e-9
                X_new[f'VWAP_PROXY_{w}'] = np.sum(rv*vv, axis=1) / np.sum(vv, axis=1)
        if 'MEDIAN_DAILY_TURNOVER' in X.columns and 'SIGNED_VOLUME_1' in X.columns:
            X_new['VOL_TURNOVER_RATIO'] = np.abs(X['SIGNED_VOLUME_1']) / (X['MEDIAN_DAILY_TURNOVER']+1e-9)
        return X_new

class CrossLagCorrelationGenerator(FeatureGenerator):
    def __init__(self, windows=[10,20]): self.windows = windows
    def transform(self, X):
        X_new = pd.DataFrame(index=X.index)
        for w in self.windows:
            ar = [f'RET_{i}' for i in range(1, w+1) if f'RET_{i}' in X.columns]
            av = [f'SIGNED_VOLUME_{i}' for i in range(1, w+1) if f'SIGNED_VOLUME_{i}' in X.columns]
            if len(ar)==w and len(av)==w:
                rv, vv = X[ar].values, X[av].values
                rd, vd = rv - rv.mean(axis=1, keepdims=True), vv - vv.mean(axis=1, keepdims=True)
                num = np.sum(rd*vd, axis=1)
                den = np.sqrt(np.sum(rd**2, axis=1) * np.sum(vd**2, axis=1)) + 1e-9
                X_new[f'CORR_RET_VOL_{w}'] = num / den
            if len(ar)==w and w >= 6:
                half = w // 2
                recent, older = X[ar[:half]].values, X[ar[half:2*half]].values
                rd = recent - recent.mean(axis=1, keepdims=True)
                od = older - older.mean(axis=1, keepdims=True)
                num = np.sum(rd*od, axis=1)
                den = np.sqrt(np.sum(rd**2, axis=1) * np.sum(od**2, axis=1)) + 1e-9
                X_new[f'AUTOCORR_RET_{w}'] = num / den
        return X_new

print('Feature generators loaded ✅')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. Autoencoder (PyTorch GPU)
# ══════════════════════════════════════════════════════════════

class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout * 0.7),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, input_dim),
        )
    def forward(self, x): return self.decoder(self.encoder(x))
    def encode(self, x): return self.encoder(x)


def train_autoencoder(X, latent_dim=16, lr=1e-3, dropout=0.3,
                      n_epochs=100, batch_size=512, patience=10, verbose=True):
    """Train AE and return (model, latent_features)."""
    input_dim = X.shape[1]
    n = len(X); n_val = int(n * 0.1)
    idx = np.random.RandomState(42).permutation(n)
    X_tr, X_va = X[idx[n_val:]], X[idx[:n_val]]

    train_dl = DataLoader(TensorDataset(torch.FloatTensor(X_tr)), batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(TensorDataset(torch.FloatTensor(X_va)), batch_size=batch_size*2)

    model = Autoencoder(input_dim, latent_dim, dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.MSELoss()

    best_loss, wait, best_state = float('inf'), 0, None
    for ep in range(n_epochs):
        model.train()
        t_loss = 0
        for (b,) in train_dl:
            b = b.to(device)
            loss = crit(model(b), b)
            opt.zero_grad(); loss.backward(); opt.step()
            t_loss += loss.item() * len(b)
        t_loss /= len(X_tr)

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for (b,) in val_dl:
                b = b.to(device)
                v_loss += crit(model(b), b).item() * len(b)
        v_loss /= len(X_va)

        if v_loss < best_loss:
            best_loss, wait = v_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if verbose and (ep+1) % 10 == 0:
            print(f'    Epoch {ep+1:3d}: train={t_loss:.6f} val={v_loss:.6f}{" *" if wait==0 else ""}')
        if wait >= patience:
            if verbose: print(f'    Early stop at epoch {ep+1}')
            break

    if best_state: model.load_state_dict(best_state)
    model.eval()
    if verbose: print(f'  AE: latent={latent_dim}, best_val_loss={best_loss:.6f}')
    return model


def ae_transform(model, X, batch_size=2048):
    """Extract latent features from trained AE."""
    model.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            b = torch.FloatTensor(X[i:i+batch_size]).to(device)
            parts.append(model.encode(b).cpu().numpy())
    return np.concatenate(parts)

print('Autoencoder loaded ✅')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. PurgedKFold & Classifier
# ══════════════════════════════════════════════════════════════

class PurgedKFold:
    def __init__(self, n_splits=5, purge_pct=0.02):
        self.n_splits, self.purge_pct = n_splits, purge_pct
    def split(self, X, y=None, groups=None):
        ut = np.sort(np.unique(groups))
        nt = len(ut); ps = int(nt * self.purge_pct); fs = nt // self.n_splits
        for i in range(self.n_splits):
            vs, ve = i*fs, ((i+1)*fs if i < self.n_splits-1 else nt)
            val_t = set(ut[vs:ve])
            excl = set(ut[max(0,vs-ps):min(nt,ve+ps)])
            tr_t = set(ut) - excl
            tr_idx = np.where(np.isin(groups, list(tr_t)))[0]
            va_idx = np.where(np.isin(groups, list(val_t)))[0]
            if len(tr_idx) > 0 and len(va_idx) > 0:
                yield tr_idx, va_idx


def make_classifier(penalty='l2', C=1.0, l1_ratio=0.5):
    if penalty == 'elasticnet':
        return SGDClassifier(loss='log_loss', penalty='elasticnet',
                             alpha=1.0/C, l1_ratio=l1_ratio,
                             max_iter=1000, random_state=42, n_jobs=-1)
    solver = 'saga' if penalty == 'l1' else 'lbfgs'
    return LogisticRegression(penalty=penalty, C=C, solver=solver,
                              max_iter=1000, random_state=42, n_jobs=-1)

print('CV & Classifier loaded ✅')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. Load Data
# ══════════════════════════════════════════════════════════════

USE_SAMPLE = False  # Set True for quick testing

suffix = '_sample' if USE_SAMPLE else ''
X_train_raw = pd.read_csv(f'{DATA_DIR}/X_train{suffix}.csv')
y_train_raw = pd.read_csv(f'{DATA_DIR}/y_train{suffix}.csv')
X_test_raw  = pd.read_csv(f'{DATA_DIR}/X_test.csv')

train_df = X_train_raw.merge(y_train_raw, on='ROW_ID')
y = (train_df['target'] > 0).astype(int)
X_train = train_df.drop(columns=['target', 'ROW_ID'])
X_test  = X_test_raw.drop(columns=['ROW_ID'])
test_ids = X_test_raw['ROW_ID']
ts_groups = X_train['TS'].str.extract(r'(\d+)')[0].astype(int).values

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Target balance: {y.mean():.3f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. Feature Engineering (J3)
# ══════════════════════════════════════════════════════════════

generators = [
    RawFeatureGenerator(
        cols=[f'RET_{i}' for i in range(1,21)]
           + [f'SIGNED_VOLUME_{i}' for i in range(1,21)]
           + ['MEDIAN_DAILY_TURNOVER']
    ),
    RollingStatFeatureGenerator(
        cols=[f'RET_{i}' for i in range(1,21)],
        windows=[5, 10, 20],
        operations=['mean', 'std', 'min', 'max']
    ),
    MomentumGenerator(windows=[(5,20), (1,5), (1,20)]),
    VolatilityRatioGenerator(windows=[(5,20)]),
    ShortTermInteractionGenerator(max_lag=10),
    HigherOrderStatsGenerator(windows=[5, 10, 20]),
    SignFlipGenerator(windows=[5, 10, 20]),
    VolumeFlowGenerator(windows=[5, 10]),
    CrossLagCorrelationGenerator(windows=[10, 20]),
]

def build_j3_features(X, y=None, gens=None, fit=True):
    X_feat = pd.DataFrame(index=X.index)
    for gen in gens:
        if fit and hasattr(gen, 'fit'): gen.fit(X, y)
        X_feat = pd.concat([X_feat, gen.transform(X)], axis=1)
    return X_feat.loc[:, ~X_feat.columns.duplicated()]

X_train_feat = build_j3_features(X_train, y, generators, fit=True)
X_test_feat  = build_j3_features(X_test, generators=generators, fit=False)
print(f'Features generated: {X_train_feat.shape[1]}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. Imputation + Scaling
# ══════════════════════════════════════════════════════════════

imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train_feat)
X_test_imp  = imputer.transform(X_test_feat)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

input_dim = X_train_scaled.shape[1]
print(f'Scaled: {input_dim} features, {X_train_scaled.shape[0]} rows')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. Optuna Optimization
# ══════════════════════════════════════════════════════════════

N_TRIALS = 30  # Increase for better results

def objective(trial):
    pca_n = trial.suggest_int('pca_n', 10, min(input_dim, 60))
    ae_latent = trial.suggest_int('ae_latent', 8, 32)
    ae_lr = trial.suggest_float('ae_lr', 1e-4, 1e-2, log=True)
    ae_dropout = trial.suggest_float('ae_dropout', 0.1, 0.5)

    # PCA
    pca = PCA(n_components=min(pca_n, input_dim), random_state=42)
    X_pca = pca.fit_transform(X_train_scaled)

    # Autoencoder (GPU)
    ae_model = train_autoencoder(X_train_scaled, latent_dim=ae_latent,
                                 lr=ae_lr, dropout=ae_dropout, verbose=False)
    X_ae = ae_transform(ae_model, X_train_scaled)

    X_combined = np.hstack([X_pca, X_ae])

    # Classifier
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
    C = trial.suggest_float('C', 1e-3, 100.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9) if penalty == 'elasticnet' else 0.5

    # PurgedKFold CV
    pkf = PurgedKFold(n_splits=3, purge_pct=0.02)
    val_scores, train_scores = [], []
    y_arr = y.values
    for tr_idx, va_idx in pkf.split(X_combined, groups=ts_groups):
        clf = make_classifier(penalty, C, l1_ratio)
        clf.fit(X_combined[tr_idx], y_arr[tr_idx])
        val_scores.append(accuracy_score(y_arr[va_idx], clf.predict(X_combined[va_idx])))
        train_scores.append(accuracy_score(y_arr[tr_idx], clf.predict(X_combined[tr_idx])))

    val_acc = np.mean(val_scores)
    trial.set_user_attr('train_acc', np.mean(train_scores))
    trial.set_user_attr('gap', np.mean(train_scores) - val_acc)
    return val_acc

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

bp = study.best_params
print(f'\n✅ Best trial: #{study.best_trial.number}')
print(f'   Val accuracy: {study.best_value:.4f}')
print(f'   Gap: {study.best_trial.user_attrs["gap"]:.4f}')
print(f'   Params: {bp}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. Build Final Features & Evaluate (5-fold)
# ══════════════════════════════════════════════════════════════

# Rebuild with best hyperparams
pca_final = PCA(n_components=min(bp['pca_n'], input_dim), random_state=42)
X_train_pca = pca_final.fit_transform(X_train_scaled)
X_test_pca  = pca_final.transform(X_test_scaled)

ae_final = train_autoencoder(X_train_scaled, latent_dim=bp['ae_latent'],
                              lr=bp['ae_lr'], dropout=bp['ae_dropout'], verbose=True)
X_train_ae = ae_transform(ae_final, X_train_scaled)
X_test_ae  = ae_transform(ae_final, X_test_scaled)

X_train_final = np.hstack([X_train_pca, X_train_ae])
X_test_final  = np.hstack([X_test_pca, X_test_ae])

print(f'Final features: Train {X_train_final.shape}, Test {X_test_final.shape}')
print(f'PCA variance explained: {pca_final.explained_variance_ratio_.sum()*100:.1f}%')

# 5-fold evaluation
l1_ratio = bp.get('l1_ratio', 0.5)
pkf = PurgedKFold(n_splits=5, purge_pct=0.02)
val_s, tr_s = [], []
y_arr = y.values
for tr_i, va_i in pkf.split(X_train_final, groups=ts_groups):
    clf = make_classifier(bp['penalty'], bp['C'], l1_ratio)
    clf.fit(X_train_final[tr_i], y_arr[tr_i])
    val_s.append(accuracy_score(y_arr[va_i], clf.predict(X_train_final[va_i])))
    tr_s.append(accuracy_score(y_arr[tr_i], clf.predict(X_train_final[tr_i])))

print(f'\n5-Fold Purged CV:')
print(f'  Train: {np.mean(tr_s):.4f} | Val: {np.mean(val_s):.4f} (±{np.std(val_s):.4f}) | Gap: {np.mean(tr_s)-np.mean(val_s):.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. Train Final Model & Generate Submission
# ══════════════════════════════════════════════════════════════

clf_final = make_classifier(bp['penalty'], bp['C'], l1_ratio)
clf_final.fit(X_train_final, y)

# Get probabilities
if hasattr(clf_final, 'predict_proba'):
    probs = clf_final.predict_proba(X_test_final)[:, 1]
else:
    dec = clf_final.decision_function(X_test_final)
    probs = 1 / (1 + np.exp(-dec))

# Calibrate threshold
target_ratio = y.mean()
sorted_probs = np.sort(probs)[::-1]
n_pos = int(len(probs) * target_ratio)
threshold = sorted_probs[min(n_pos, len(sorted_probs)-1)]
preds = (probs > threshold).astype(int)

print(f'Threshold: {threshold:.6f} (target ratio: {target_ratio:.4f})')
print(f'Predictions: {preds.sum()} pos / {len(preds)} total ({preds.mean()*100:.1f}%)')

# Save
submission = pd.DataFrame({'ROW_ID': test_ids, 'score': preds})
output_file = 'submission_j4_hybrid_pca_ae_logreg.csv'
submission.to_csv(output_file, index=False)
print(f'\n✅ Saved: {output_file}')

# Download from Colab
from google.colab import files
files.download(output_file)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. Top Trials Recap
# ══════════════════════════════════════════════════════════════

trials_df = study.trials_dataframe().sort_values('value', ascending=False).head(10)
for _, row in trials_df.iterrows():
    print(f'Trial {int(row["number"]):>2}: val={row["value"]:.4f}, gap={row["user_attrs_gap"]:.4f}')